# Few-shot Example Generation

## Overview

This notebook generates the final few-shot demonstration examples used throughout the evaluation.

The selected representative examples obtained from the previous notebook are converted into prompts using the prompt builder. These prompts are analyzed by an expert LLM to produce high-quality demonstrations.

The finalized demonstrations are saved as reusable prompt templates for the evaluation pipeline.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / "datasets"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
GENERATED_DIR = PROJECT_ROOT / "generated_prompts"

print("Project Root :", PROJECT_ROOT)

Project Root : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework


In [2]:
from utils.prompt_builder import build_prompt

print("Prompt builder imported successfully.")

Prompt builder imported successfully.


### load data set

In [3]:
DATASET_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "context_augmented_dataset_preprocessed.jsonl"
)

working_df = pd.read_json(
    DATASET_PATH,
    lines=True
)

working_df["isFlaky"] = working_df["isFlaky"].astype(bool)

print("=" * 80)
print("DATASET LOADED")
print("=" * 80)
print(f"Total Records : {len(working_df)}")

DATASET LOADED
Total Records : 2743


## Load Selected Examples

The representative examples selected in the previous notebook are loaded from the preprocessed dataset.

These examples are used to construct the final few-shot demonstrations for both the code-only and context-augmented prompting strategies.

In [4]:
FLAKY_INDEX = 758
NON_FLAKY_INDEX = 1951

flaky_example = working_df.loc[FLAKY_INDEX].to_dict()
non_flaky_example = working_df.loc[NON_FLAKY_INDEX].to_dict()

print("✓ Selected examples loaded.")

✓ Selected examples loaded.


### verify selection

In [5]:
working_df.loc[
    [FLAKY_INDEX, NON_FLAKY_INDEX],
    [
        "test_id",
        "isFlaky",
        "issue_category"
    ]
]

,test_id,isFlaky,issue_category
758,OpenRefinemaina68ba3btestSelectedEmptyChoice,True,Implementation Dependent
1951,JacksonCore-22-1,False,Non-Flaky


## Generate Prompts

The selected examples are converted into prompts using the prompt builder.

For each example, two prompt variants are generated:

- Code-only
- Context-Augmented

These prompts are submitted to an expert LLM to obtain high-quality demonstrations.

### Generate Code-Only Prompts

In [6]:
flaky_code_prompt = build_prompt(
    sample=flaky_example,
    strategy="zero_shot",
    include_context=False
)

non_flaky_code_prompt = build_prompt(
    sample=non_flaky_example,
    strategy="zero_shot",
    include_context=False
)

print("✓ Code-only prompts generated.")

✓ Code-only prompts generated.


### Create Output Directory

In [7]:
PROMPT_OUTPUT_DIR = (
    PROJECT_ROOT
    / "generated_prompts"
)

PROMPT_OUTPUT_DIR.mkdir(
    exist_ok=True
)

PROMPT_OUTPUT_DIR

WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/generated_prompts')

### Generate Context Prompts

In [8]:
flaky_context_prompt = build_prompt(
    sample=flaky_example,
    strategy="zero_shot",
    include_context=True
)

non_flaky_context_prompt = build_prompt(
    sample=non_flaky_example,
    strategy="zero_shot",
    include_context=True
)

print("✓ Context prompts generated.")

✓ Context prompts generated.


### Save All Prompts

In [9]:
prompt_files = {
    "flaky_code_prompt.txt": flaky_code_prompt,
    "flaky_context_prompt.txt": flaky_context_prompt,
    "non_flaky_code_prompt.txt": non_flaky_code_prompt,
    "non_flaky_context_prompt.txt": non_flaky_context_prompt,
}

for filename, content in prompt_files.items():

    with open(
        PROMPT_OUTPUT_DIR / filename,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(content)

print("✓ Prompt files saved.")

for filename in prompt_files:
    print(filename)

✓ Prompt files saved.
flaky_code_prompt.txt
flaky_context_prompt.txt
non_flaky_code_prompt.txt
non_flaky_context_prompt.txt


## Generate Expert Demonstrations

The generated prompts are submitted to an expert Large Language Model to obtain high-quality reference demonstrations.

The expert model analyzes each prompt and returns the expected classification, flaky category (if applicable), reasoning, and supporting evidence.

These expert-generated responses are manually reviewed to ensure correctness and consistency before being incorporated into the final few-shot prompt templates.

In [10]:
print("=" * 80)
print("FLAKY - CODE ONLY")
print("=" * 80)
print(flaky_code_prompt[:3000])

FLAKY - CODE ONLY
You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the execution order of tests 

In [11]:
print("=" * 80)
print("FLAKY - CONTEXT")
print("=" * 80)
print(flaky_context_prompt[:3000])

FLAKY - CONTEXT
You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the execution order of tests be

In [12]:
print("=" * 80)
print("NON-FLAKY - CODE ONLY")
print("=" * 80)
print(non_flaky_code_prompt[:3000])

NON-FLAKY - CODE ONLY
You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the execution order of te

In [13]:
print("=" * 80)
print("NON-FLAKY - CONTEXT")
print("=" * 80)
print(non_flaky_context_prompt[:3000])

NON-FLAKY - CONTEXT
You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the execution order of test

## Expert Verification

Representative candidate examples were submitted to an expert LLM to verify that they could be correctly classified using the intended prompt configuration.

An example was accepted only if:
- its expert-generated classification matched the dataset label; and
- the reasoning was consistent with the available evidence.

Candidates that failed this verification step were replaced before constructing the final few-shot demonstrations.

## Initial Validation Results

The generated expert responses were compared against the corresponding dataset labels.

The non-flaky example was correctly classified under both prompt configurations. However, the initial flaky example was incorrectly classified as **Non-Flaky** when only the test code was provided.

Since few-shot demonstrations must accurately reflect the dataset annotations, the initial flaky demonstration was rejected.

A replacement flaky example is therefore selected using the next highest-ranked representative candidate identified in the example selection process.

In [14]:
# ============================================================
# Load Replacement Flaky Example
# ============================================================

REPLACEMENT_FLAKY_INDEX = 789

replacement_flaky_example = working_df.loc[
    REPLACEMENT_FLAKY_INDEX
]

print("=" * 80)
print("Replacement Flaky Example")
print("=" * 80)

display(
    working_df.loc[
        [REPLACEMENT_FLAKY_INDEX],
        ["test_id", "issue_category", "isFlaky"]
    ]
)

Replacement Flaky Example


,test_id,issue_category,isFlaky
789,jolokiasupportspring275165bsystemProperties,Implementation Dependent,True


## Generate Replacement Prompts

The replacement flaky example is converted into both code-only and context-augmented prompts using the same prompt templates.

The previously validated non-flaky example remains unchanged.

In [15]:
replacement_flaky_code_prompt = build_prompt(
    sample=replacement_flaky_example,
    strategy="zero_shot",
    include_context=False
)

replacement_flaky_context_prompt = build_prompt(
    sample=replacement_flaky_example,
    strategy="zero_shot",
    include_context=True
)

In [16]:
with open(
    PROMPT_OUTPUT_DIR / "replacement_flaky_code_prompt.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(replacement_flaky_code_prompt)

with open(
    PROMPT_OUTPUT_DIR / "replacement_flaky_context_prompt.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(replacement_flaky_context_prompt)

print("✓ Replacement prompts saved.")

✓ Replacement prompts saved.


## Expert Evaluation (Replacement Candidate)

Submit the replacement prompts to the expert language model.

After obtaining the responses, save them in the `generated_prompts` directory as:

- `response_replacement_flaky_code_prompt.txt`
- `response_replacement_flaky_context_prompt.txt`

## Validation Results - Failed

Results is failed due to label mismatch from classification response label

## Third Candidate Selection

Following the refinement of the demonstration selection strategy, a third flaky candidate is selected.

Unlike the previous candidates, this example is chosen because the flaky behaviour is explicitly observable within the test code through concurrency- or timing-related constructs. This improves the suitability of the example as a code-only demonstration while maintaining consistency with the dataset annotations.

In [17]:
THIRD_FLAKY_TEST_ID = "CURATOR-671"

working_df.loc[
    working_df["test_id"] == THIRD_FLAKY_TEST_ID,
    ["test_id", "issue_category", "isFlaky"]
]

,test_id,issue_category,isFlaky


In [18]:
working_df[
    working_df["test_id"].str.contains("CURATOR", na=False)
][["test_id", "issue_category"]]

,test_id,issue_category
1081,CURATOR-681,Time Dependent


In [19]:
THIRD_FLAKY_TEST_ID = "CURATOR-681"

third_flaky_example = working_df[
    working_df["test_id"] == THIRD_FLAKY_TEST_ID
].iloc[0].to_dict()

display(
    pd.DataFrame([
        {
            "test_id": third_flaky_example["test_id"],
            "issue_category": third_flaky_example["issue_category"],
            "isFlaky": third_flaky_example["isFlaky"]
        }
    ])
)

,test_id,issue_category,isFlaky
0,CURATOR-681,Time Dependent,True


In [20]:
# ============================================================
# Generate Code-Only Prompt for Third Candidate
# ============================================================

third_flaky_code_prompt = build_prompt(
    sample=third_flaky_example,
    strategy="zero_shot",
    include_context=False
)

print("✓ Code-only prompt generated successfully.")

✓ Code-only prompt generated successfully.


In [21]:
# ============================================================
# Save Code-Only Prompt
# ============================================================

third_prompt_path = (
    PROMPT_OUTPUT_DIR
    / "third_flaky_code_prompt.txt"
)

with open(third_prompt_path, "w", encoding="utf-8") as f:
    f.write(third_flaky_code_prompt)

print(f"✓ Prompt saved to:\n{third_prompt_path}")

✓ Prompt saved to:
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\third_flaky_code_prompt.txt


In [22]:
# ============================================================
# Display Generated Prompt
# ============================================================

print("=" * 100)
print("THIRD CANDIDATE - CODE ONLY PROMPT")
print("=" * 100)

print(third_flaky_code_prompt)

THIRD CANDIDATE - CODE ONLY PROMPT
You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the executio

## Expert Validation - Third Candidate

The generated code-only prompt is submitted to the expert language model.

The candidate is accepted only if the expert-generated classification agrees with the ground-truth dataset annotation.

In [23]:
# ============================================================
# Load Expert Response
# ============================================================

response_path = (
    PROMPT_OUTPUT_DIR
    / "response_third_flaky_code_prompt.txt"
)

with open(response_path, "r", encoding="utf-8") as f:
    third_flaky_response = f.read()

print("✓ Expert response loaded successfully.")

✓ Expert response loaded successfully.


In [24]:
# ============================================================
# Display Expert Response
# ============================================================

print("=" * 100)
print("EXPERT RESPONSE - THIRD CANDIDATE")
print("=" * 100)

print(third_flaky_response)

EXPERT RESPONSE - THIRD CANDIDATE
{
"classification": "Flaky",
"category": "Time Dependent",
"reasoning": "The test relies on fixed sleep intervals (Thread.sleep(100)) and busy-wait polling on semaphore permits to coordinate synchronization between concurrently executing threads. The correctness of the test's assertions depends on threads completing their sleep/wait cycles within expected time windows relative to each other, making the outcome sensitive to scheduling delays, system load, or JVM thread scheduling variance rather than deterministic control flow.",
"evidence": [
"Test Code"
]
}


## Final Validation Results

The third candidate was successfully classified by the expert language model using only the test code.

Unlike the previous representative examples, the explicit concurrency- and timing-related constructs present in the test implementation enabled the expert model to correctly identify the flaky behaviour without requiring additional contextual information.

Therefore, this candidate is accepted as the final flaky demonstration for the code-only prompt configuration.

In [25]:
validation_summary = pd.DataFrame(
    [
        {
            "Attempt": 1,
            "Candidate": "OpenRefine",
            "Selection Strategy": "Representative",
            "Expert Result": "Rejected"
        },
        {
            "Attempt": 2,
            "Candidate": "Jolokia",
            "Selection Strategy": "Representative",
            "Expert Result": "Rejected"
        },
        {
            "Attempt": 3,
            "Candidate": THIRD_FLAKY_TEST_ID,
            "Selection Strategy": "Observable Flaky Indicators",
            "Expert Result": "Accepted"
        }
    ]
)

display(validation_summary)

,Attempt,Candidate,Selection Strategy,Expert Result
0,1,OpenRefine,Representative,Rejected
1,2,Jolokia,Representative,Rejected
2,3,CURATOR-681,Observable Flaky Indicators,Accepted


## Final Demonstration Examples

The final demonstrations used throughout the experiments are:

- Flaky (Code-only): Third validated candidate
- Non-Flaky (Code-only): Previously validated representative example
- Flaky (Context): Representative example
- Non-Flaky (Context): Representative example

These examples are used to construct the few-shot demonstrations employed during evaluation.

In [26]:
final_examples = {
    "code_flaky": third_flaky_example,
    "code_non_flaky": non_flaky_example,

    "context_flaky": third_flaky_example,
    "context_non_flaky": non_flaky_example
}

In [27]:
# Code-only
final_flaky_code_prompt = build_prompt(
    sample=final_examples["code_flaky"],
    strategy="zero_shot",
    include_context=False
)

final_non_flaky_code_prompt = build_prompt(
    sample=final_examples["code_non_flaky"],
    strategy="zero_shot",
    include_context=False
)

# Context
final_flaky_context_prompt = build_prompt(
    sample=final_examples["context_flaky"],
    strategy="zero_shot",
    include_context=True
)

final_non_flaky_context_prompt = build_prompt(
    sample=final_examples["context_non_flaky"],
    strategy="zero_shot",
    include_context=True
)

## save examples

In [28]:
final_prompt_dir = PROMPT_OUTPUT_DIR / "final"

final_prompt_dir.mkdir(parents=True, exist_ok=True)

prompt_files = {
    "flaky_code_prompt.txt": final_flaky_code_prompt,
    "non_flaky_code_prompt.txt": final_non_flaky_code_prompt,
    "flaky_context_prompt.txt": final_flaky_context_prompt,
    "non_flaky_context_prompt.txt": final_non_flaky_context_prompt
}

for filename, content in prompt_files.items():
    with open(final_prompt_dir / filename, "w", encoding="utf-8") as f:
        f.write(content)

print("✓ Final prompt files saved.")

✓ Final prompt files saved.


In [29]:
# ============================================================
# Selected Few-Shot Examples
# ============================================================

selected_test_ids = sorted({
    example["test_id"]
    for example in final_examples.values()
})

print("=" * 80)
print("Selected Few-Shot Examples")
print("=" * 80)

display(
    working_df[
        working_df["test_id"].isin(selected_test_ids)
    ][
        ["test_id", "issue_category", "isFlaky"]
    ]
)

Selected Few-Shot Examples


,test_id,issue_category,isFlaky
1081,CURATOR-681,Time Dependent,True
1951,JacksonCore-22-1,Non-Flaky,False


In [30]:
# ============================================================
# Create Evaluation Dataset
# ============================================================

evaluation_df = working_df[
    ~working_df["test_id"].isin(selected_test_ids)
].copy()

evaluation_dataset_path = (
    PROJECT_ROOT
    / "datasets"
    / "evaluation_dataset.jsonl"
)

evaluation_df.to_json(
    evaluation_dataset_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("=" * 80)
print("Evaluation Dataset Created")
print("=" * 80)
print(f"Original Records   : {len(working_df)}")
print(f"Removed Examples   : {len(selected_test_ids)}")
print(f"Evaluation Records : {len(evaluation_df)}")
print(f"Saved to           : {evaluation_dataset_path}")

Evaluation Dataset Created
Original Records   : 2743
Removed Examples   : 2
Evaluation Records : 2741
Saved to           : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\datasets\evaluation_dataset.jsonl
